In [1]:
import ast
import os
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm
import pickle
from shutil import copyfile

In [2]:
table_dir = '../data/data_in_sep_pkl/'
assert os.path.exists(table_dir)

In [3]:
paths = []
for f in os.listdir(table_dir):
    paths.append(os.path.join(table_dir, f))
paths = sorted(paths)
tot = len(paths)
print(tot)
print(paths[0])
#print(paths.index('/home/scai/msr/aiy217586/properties/dataset_merge/table_data_old/S0022309301007542__1'))

1021
../data/data_in_sep_pkl/S0022309300000612__0


In [4]:
def get_json_dict(i):
    if os.path.exists(os.path.join(paths[i], 'unit.pkl')):
        d = pickle.load(open(os.path.join(paths[i], 'unit.pkl'), 'rb'))
        return d
    else:
        d = pickle.load(open(os.path.join(paths[i], 'init.pkl'), 'rb'))
        table_name = d['pii'] + '__' + str(d['t_idx'])
        sys.exit(f'Unit havent been extracted for {table_name}')

In [5]:
d_all_units = {}
d_units_papers = {}
prop_ids = [510, 1140, 2010, 2051, 540, 50, 180, 60, 160, 1014, 1015, 3012, 3174, 1116, 1119, 1020, 1011, 1118, 70, 1306]
prop_names = ['Density', 'GlassTransitionTg', 'RefractiveIndex', 'AbbeValue', 'YoungsModulus', 'ShearModulus', 'VickersHardness', 'PoissonRatio', 'FractureToughness', 'CrystallizationTemp', 'MeltingTemp', 'ElectricConduct', 'DielectricConst', 'TSofP', 'TAnnealingP', 'ExpansionCoeff', 'LiquidusTemperature', 'SofP', 'BulkModulus', 'ActivationEnergy']    

def norm_units_estimation(i):
    d = get_json_dict(i)
    #print(d)
    table_name = d['pii'] + '__' + str(d['t_idx'])
#     print(f'Table name = {table_name}')
    r, c = d['num_rows'], d['num_cols']
    global prop_ids, prop_names, d_all_units, d_units_papers
    
#     print(f'Comp table: {d["comp_table"]}')
#     print(f'Prop table: {d["prop_table"]}')
#     print(f'Prop names: {d["prop_names"]}')
#     print(f'Prop orient: {d["prop_orient"]}')
#     # print(d)
#     print(f'Prop index: {d["prop_row_col_indexes"]}')
    # print(f'Props: {d["props"]}')
    
    #co = 0
    
    for ind, single_prop_table in enumerate(d["props"]):
        if len(d["prop_row_col_indexes"][ind]) != 0:
            prop_name = prop_names[ind]
            #display(pd.DataFrame(single_prop_table))
            
            if d["prop_orient"] =='col':
                for indd in d["prop_row_col_indexes"][ind]:
                    #print(f'indd = {indd}')
                    s_p_t = np.array(single_prop_table)
                    #display(pd.DataFrame(np.array(s_p_t[:,indd])))
                    for ele in s_p_t[:,indd]:
                        #print(ele)
                        if len(ele)>0:
                            elem= ele[0]
                            if prop_name == 'RefractiveIndex' and elem[-1]=='':
                                print(d['anno_type'])
                            try:
                                d_all_units[prop_name].add(elem[-1])
                            except KeyError:
                                d_all_units[prop_name] = {elem[-1]}
                                
                            try:
                                d_units_papers[elem[-1]].add(table_name)
                            except KeyError:
                                d_units_papers[elem[-1]] = {table_name}
                            
                            
            elif d["prop_orient"]=='row':
                for indd in d["prop_row_col_indexes"][ind]:
                    #print(f'indd = {indd}')
                    s_p_t = np.array(single_prop_table)
                    #display(pd.DataFrame(np.array(s_p_t[indd,:])))
                    for ele in s_p_t[indd,:]:
                        #print(ele)
                        if len(ele)>0:
                            elem= ele[0]
                            if prop_name == 'RefractiveIndex' and elem[-1]=='':
                                print(elem)
                                print(d['anno_type'])
                            try:
                                d_all_units[prop_name].add(elem[-1])
                            except KeyError:
                                d_all_units[prop_name] = {elem[-1]}
                            try:
                                d_units_papers[elem[-1]].add(table_name)
                            except KeyError:
                                d_units_papers[elem[-1]] = {table_name}
            
            #co += 1
            
    return

In [6]:
for i in range(tot):
    #print(i)
    norm_units_estimation(i)

/home/scai/phd/aiz217586/.conda/envs/discomat/lib/python3.7/site-packages/ipykernel_launcher.py:32: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
/home/scai/phd/aiz217586/.conda/envs/discomat/lib/python3.7/site-packages/ipykernel_launcher.py:54: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.


In [7]:
d_units_freq = {}
for key in d_units_papers.keys():
    try:
        d_units_freq[key].add(len(d_units_papers[key]))
    except:
        d_units_freq[key] = {len(d_units_papers[key])}

In [8]:
#value to be normalized to:
den_unit = {'g/cm3': {'gm/cm3', 'g cm-3', 'g/cm-3', 'g/cm3', 'g/cm 3', 'gcm -3', 'gcm-3', 'g/cc', 'gm/cc', 'gw/cm3', 'gm cc-1', 'g/ cm3', 'g/mL', 'd gcm-3'},
                'mg/m3' : {'Mg/m3', 'mgm-3'}, 'kg/m3' : {'kgm -3', 'kgm-3', 'kg/m3', 'kg m-3', 'g/l'}, 'g/cm' : {'g/cm -1', 'gcm -1', 'gcm-1', 'g cm-1', 'g/cm'}, 
            'lb/in3': {'lb/in3', 'lb/ in3', 'lb in-3', 'lbin -3'}}

tg_unit = {'degC' : {'degC', 'degC/min', 'C'}, 'K': {'K', 'K min-1', 'T/K'}}


ri_unit = {None : {None}}


ym_unit = {'GPa': {'GPa'}, 'MPa':{'Mpa'}, 'Pa':{'Pa'}, 'psi':{'psi'}}


vh_unit = {'Kg/mm2': {'Kgf/mm 2', 'Kg/mm2', 'kgfmm-2', 'kg/mm2', 'kgmm -2'}, 'GPa': {'GPa'}, 'MPa':{'Mpa'}, 'Pa':{'Pa'}, 'psi':{'psi'}}

ft_unit = {'MPam1/2': {'MPa m', 'MPa m1/2', 'MPam1/2', 'MPa/sqrt(m)'}}

ec_unit = {'O-1 cm-1': {'O-1 cm- 1', 'O-1 cm-1', 'ohm-1 cm-1', 'Scm-1', 'S cm-1', 'S/cm'}, 'O-1 m-1': {'O-1 m- 1', 'O-1 m-1', 'ohm-1 m-1', 'Sm-1', 'S m-1', 'S/m'}}

exc_unit = {'degC-1': {'degC', 'C-1', 'x10-7/degC'}, 'K-1':{'K-1', 'x10 K-1'}}

ea_unit = {'kJ/mol': {'kJmol-1', 'kJ mol-1','kJ/mol', 'kJ / mol', 'kJ/molK'}, 'eV/at':{'eV/at'}, 'kcal/mol':{'kcal/mol', 'kcalmol-1', 'kcal mol-1'}, 'J/mol': {'Jmol-1', 'J mol-1','J/mol', 'J / mol'}, 'eV':{'eV'}}

all_train_data_with_units_pre = []

In [9]:
#prop_names = ['Density', 'GlassTransitionTg', 'RefractiveIndex', 'AbbeValue', 'YoungsModulus', 'ShearModulus', 
#'VickersHardness', 'PoissonRatio', 'FractureToughness', 'CrystallizationTemp', 'MeltingTemp', 'ElectricConduct', 
#'DielectricConst', 'TSofP', 'TAnnealingP', 'ExpansionCoeff', 'LiquidusTemperature', 'SofP', 'BulkModulus', 'ActivationEnergy']

def normalize_unit(unit, prop_name):
    global den_unit, tg_unit, ri_unit, ym_unit, vh_unit, ft_unit, ec_unit, exc_unit, ea_unit
    
    if prop_name == 'Density':
        for key, value in den_unit.items():
            if unit in value:
                return key
        if 'cm' in unit: return 'g/cm3'
        elif 'kg' in unit: return 'kg/m3'
        return unit
    
    elif prop_name in ['GlassTransitionTg', 'CrystallizationTemp', 'MeltingTemp', 'TSofP', 'TAnnealingP', 'LiquidusTemperature','SofP']:
        for key, value in tg_unit.items():
            if unit in value:
                return key
        return unit
    
    elif prop_name in ['RefractiveIndex', 'AbbeValue', 'PoissonRatio','DielectricConst']:
        return None
    
    elif prop_name in ['YoungsModulus', 'ShearModulus', 'BulkModulus']:
        for key, value in ym_unit.items():
            if unit in value:
                return key
        return unit
        
    elif prop_name == 'VickersHardness':
        for key, value in vh_unit.items():
            if unit in value:
                return key
        return unit
    
    elif prop_name == 'FractureToughness':
        for key, value in ft_unit.items():
            if unit in value:
                return key
        return unit
    
    elif prop_name == 'ElectricConduct':
        for key, value in ec_unit.items():
            if unit in value:
                return key
        return unit
    
    elif prop_name == 'ExpansionCoeff':
        for key, value in exc_unit.items():
            if unit in value:
                return key
        return unit
    
    elif prop_name == 'ActivationEnergy':
        for key, value in ea_unit.items():
            if unit in value:
                return key
        return unit
    
    else:
        sys.exit('Uncovered property')

In [10]:
def norm_units(i):
    global all_train_data_with_units_pre
    d = get_json_dict(i)
    r, c = d['num_rows'], d['num_cols']
    global prop_ids, prop_names
    table_name = d['pii'] + '___' + str(d['t_idx'])
    
#     print(f'Comp table: {d["comp_table"]}')
#     print(f'Prop table: {d["prop_table"]}')
#     print(f'Prop names: {d["prop_names"]}')
#     print(f'Prop orient: {d["prop_orient"]}')
#     # print(d)
#     print(f'Prop index: {d["prop_row_col_indexes"]}')
    # print(f'Props: {d["props"]}')
    
    #co = 0
    
    for ind, single_prop_table in enumerate(d["props"]):
        #display(pd.DataFrame(single_prop_table))
        if len(d["prop_row_col_indexes"][ind]) != 0:
            prop_name = prop_names[ind]
            #display(pd.DataFrame(single_prop_table))
            
            if d["prop_orient"]=='col':
                for prop_col in d["prop_row_col_indexes"][ind]:
                    #print(f'indd = {indd}')
                    s_p_t = np.array(single_prop_table)
                    #display(pd.DataFrame(np.array(s_p_t[:,indd])))
                    for row_n, ele in enumerate(s_p_t[:,prop_col]):
                        new_ele = []
                        for elem in ele:
                            elem = list(elem)
                            unit = elem[-1]
                            n_unit = normalize_unit(unit, prop_name)
                            elem[-1] = n_unit
                            new_ele.append(tuple(elem))
                            
                        d['props'][ind][row_n][prop_col] = new_ele
                            
                    
                            
            elif d["prop_orient"]=='row':
                
                for prop_row in d["prop_row_col_indexes"][ind]:
                    #print(f'indd = {indd}')
                    s_p_t = np.array(single_prop_table)
                    #display(pd.DataFrame(np.array(s_p_t[:,indd])))
                    for col_n, ele in enumerate(s_p_t[prop_row,:]):
                        new_ele = []
                        for elem in ele:
                            elem  = list(elem)
                            unit = elem[-1]
                            n_unit = normalize_unit(unit, prop_name)
                            elem[-1] = n_unit
                            new_ele.append(tuple(elem))
                            
                        d['props'][ind][prop_row][col_n] = new_ele
            
            #co += 1
        #display(pd.DataFrame(single_prop_table))
        
    pickle.dump(d, open(os.path.join(paths[i], 'new_norm.pkl'), 'wb'))
    all_train_data_with_units_pre.append(d)
    
    return

In [11]:
for i in range(tot):
    norm_units(i)
pickle.dump(all_train_data_with_units_pre, open(os.path.join('../data/', "prop_train_data_with_units.pkl"), 'wb'))
print(len(all_train_data_with_units_pre))

/home/scai/phd/aiz217586/.conda/envs/discomat/lib/python3.7/site-packages/ipykernel_launcher.py:27: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
/home/scai/phd/aiz217586/.conda/envs/discomat/lib/python3.7/site-packages/ipykernel_launcher.py:46: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.


1021
